1. Install and Import

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu128
True


In [2]:
import sys

!conda install -y -c conda-forge pyarrow
!{sys.executable} -m pip install transformers evaluate accelerate
!{sys.executable} -m pip install datasets --no-build-isolation

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/pytorch

  added / updated specs:
    - pyarrow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aws-crt-cpp-0.40.1         |       h102d43b_3         407 KB  conda-forge
    aws-sdk-cpp-1.11.833       |       hc7390e0_9         3.2 MB  conda-forge
    azure-core-cpp-1.16.3      |       h206d751_0         341 KB  conda-forge
    azure-identity-cpp-1.13.3  |       h71f81a8_2         245 KB  conda-forge
    azure-storage-blobs-cpp-12.18.0|       h74b55db_1         576 KB  conda-forge
    azure-storage-common-cpp-12.14.0|       hf596fc9_1         156 KB  conda-forge
    azure-storage-files-datalake-cpp-12.16.0|       h1f05bef_1         301 KB  conda-forge
    gflags-2.3.1               |       h54a6638_0         128 KB  co

In [3]:
import pyarrow
import datasets
import transformers
import torch

print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

pyarrow: 25.0.0
datasets: 5.0.1
transformers: 5.15.1
torch: 2.10.0+cu128


In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset

2. Set random seed for reproducibility

In [5]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

3. Load pairwise gold dataset with bins

In [6]:
# Load the same dataset with fixed Bin assignments
df = pd.read_csv(
    "pairwise_gold_dataset_with_bins.csv",
    encoding="utf-8-sig"
)

# Check label distribution within each bin
print(df["Label"].value_counts())
print(pd.crosstab(df["Bin"], df["Label"]))

Label
0    2830
1     566
Name: count, dtype: int64
Label    0   1
Bin           
0      283  57
1      283  57
2      283  57
3      283  57
4      283  57
5      283  57
6      283  56
7      283  56
8      283  56
9      283  56


4. Define models

In [7]:
MODEL_LIST = {
    "BERT": "bert-base-uncased",
    "SciBERT": "allenai/scibert_scivocab_uncased"
}

5. Convert pandas DataFrame to Hugging Face Dataset

In [8]:
def make_hf_dataset(dataframe):
    # Keep only columns needed for model training
    temp_df = dataframe[["OrgName_1", "OrgName_2", "Label"]].copy()
    
    # Rename Label to labels for Hugging Face Trainer
    temp_df = temp_df.rename(columns={"Label": "labels"})
    
    return Dataset.from_pandas(temp_df, preserve_index=False)

6. Tokenization function

In [9]:
def tokenize_dataset(dataset, tokenizer, max_length=128):
    # Tokenize organization name pairs as sentence pairs
    def tokenize_batch(batch):
        return tokenizer(
            batch["OrgName_1"],
            batch["OrgName_2"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )
    
    tokenized = dataset.map(tokenize_batch, batched=True)
    
    # Set PyTorch format
    tokenized = tokenized.remove_columns(["OrgName_1", "OrgName_2"])
    tokenized.set_format("torch")
    
    return tokenized

7. Metric function for Trainer

In [10]:
def compute_metrics(eval_pred):
    # Compute positive-class and negative-class precision, recall, and F1
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1],
        average=None,
        zero_division=0
    )
    
    return {
        "nonmatch_precision": class_precision[0],
        "nonmatch_recall": class_recall[0],
        "nonmatch_f1": class_f1[0],
        "match_precision": class_precision[1],
        "match_recall": class_recall[1],
        "match_f1": class_f1[1],
    }

8. Main 10-bin evaluation function

In [11]:
def run_transformer_10bin_evaluation(df, model_label, model_name):
    fold_results = []
    pair_predictions = []
    
    for test_bin in range(10):
        # Use the next bin as development set
        dev_bin = (test_bin + 1) % 10
        
        # Use the remaining 8 bins as training set
        train_bins = [
            b for b in range(10)
            if b not in [test_bin, dev_bin]
        ]
        
        # Split data into train, development, and test sets
        train_df = df[df["Bin"].isin(train_bins)].reset_index(drop=True)
        dev_df = df[df["Bin"] == dev_bin].reset_index(drop=True)
        test_df = df[df["Bin"] == test_bin].reset_index(drop=True)
        
        print(f"\n==============================")
        print(f"{model_label} | Fold {test_bin}")
        print("==============================")
        print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
        print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
        print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())
        
        # Load tokenizer and model for each fold
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )
        
        # Convert to Hugging Face datasets
        train_dataset = make_hf_dataset(train_df)
        dev_dataset = make_hf_dataset(dev_df)
        test_dataset = make_hf_dataset(test_df)
        
        # Tokenize datasets
        train_tokenized = tokenize_dataset(train_dataset, tokenizer)
        dev_tokenized = tokenize_dataset(dev_dataset, tokenizer)
        test_tokenized = tokenize_dataset(test_dataset, tokenizer)
        
        # Output directory for this fold
        output_dir = f"./outputs_{model_label}_fold_{test_bin}"
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=5,
            weight_decay=0.01,
            logging_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_match_f1",
            greater_is_better=True,
            save_total_limit=1,
            seed=RANDOM_SEED,
            report_to="none"
        )
        
        # Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_tokenized,
            eval_dataset=dev_tokenized,
            compute_metrics=compute_metrics,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=2)
            ]
        )
        
        # Fine-tune model
        trainer.train()
        
        # Predict on test set
        test_output = trainer.predict(test_tokenized)
        logits = test_output.predictions
        y_test = test_df["Label"].values
        
        # Convert logits to probabilities
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
        y_prob_match = probs[:, 1]
        y_prob_nonmatch = probs[:, 0]
        
        # Convert probabilities to predicted labels
        y_pred = np.argmax(probs, axis=1)
        
        # Compute class-specific metrics
        class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
            y_test,
            y_pred,
            labels=[0, 1],
            average=None,
            zero_division=0
        )
        
        # Save fold-level results
        fold_results.append({
            "Fold": test_bin,
            "Model": model_label,
            
            "Match_Precision": class_precision[1],
            "Match_Recall": class_recall[1],
            "Match_F1": class_f1[1],
            "Match_Support": class_support[1],
            
            "NonMatch_Precision": class_precision[0],
            "NonMatch_Recall": class_recall[0],
            "NonMatch_F1": class_f1[0],
            "NonMatch_Support": class_support[0],
        })
        
        # Save pair-level predictions
        temp_pred_df = test_df.copy()
        temp_pred_df["Fold"] = test_bin
        temp_pred_df["Model"] = model_label
        temp_pred_df["TrueLabel"] = y_test
        temp_pred_df["PredictedLabel"] = y_pred
        temp_pred_df["Probability_Match"] = y_prob_match
        temp_pred_df["Probability_NonMatch"] = y_prob_nonmatch
        
        pair_predictions.append(temp_pred_df)
        
        # Print classification report
        print(classification_report(
            y_test,
            y_pred,
            target_names=["NonMatch", "Match"],
            digits=4,
            zero_division=0
        ))
        
        # Clear GPU memory after each fold
        del model
        del trainer
        torch.cuda.empty_cache()
    
    fold_results_df = pd.DataFrame(fold_results)
    pair_predictions_df = pd.concat(pair_predictions, ignore_index=True)
    
    return fold_results_df, pair_predictions_df

9. Run BERT and SciBERT

In [ ]:
all_fold_results = []
all_pair_predictions = []

for model_label, model_name in MODEL_LIST.items():
    fold_results_df, pair_predictions_df = run_transformer_10bin_evaluation(
        df=df,
        model_label=model_label,
        model_name=model_name
    )
    
    all_fold_results.append(fold_results_df)
    all_pair_predictions.append(pair_predictions_df)

transformer_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
transformer_pair_predictions_df = pd.concat(all_pair_predictions, ignore_index=True)


BERT | Fold 0
Train: (2716, 9) {0: 2264, 1: 452}
Dev: (340, 9) {0: 283, 1: 57}
Test: (340, 9) {0: 283, 1: 57}


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (241 outputs left out)

Map:   0%|          | 0/339 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Nonmatch Precision,Nonmatch Recall,Nonmatch F1,Match Precision,Match Recall,Match F1
1,0.161832,0.064314,0.982578,0.996466,0.989474,0.981132,0.912281,0.945455
2,0.060441,0.088449,0.982394,0.985866,0.984127,0.928571,0.912281,0.920354


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

10. Save results

In [ ]:
# Save fold-level evaluation results
transformer_fold_results_df.to_csv(
    "bert_scibert_fold_level_results_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save pair-level prediction results
transformer_pair_predictions_df.to_csv(
    "bert_scibert_pair_level_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

11. Summarize performance

In [ ]:
transformer_summary_df = (
    transformer_fold_results_df
    .groupby("Model")[
        [
            "Match_Precision",
            "Match_Recall",
            "Match_F1",
            "NonMatch_Precision",
            "NonMatch_Recall",
            "NonMatch_F1"
        ]
    ]
    .agg(["mean", "std"])
)

print(transformer_summary_df)

transformer_summary_df.to_csv(
    "bert_scibert_summary_match_nonmatch.csv",
    encoding="utf-8-sig"
)

12. Save HAC-ready similarity scores

In [ ]:
transformer_hac_score_df = transformer_pair_predictions_df[
    [
        "Model",
        "Fold",
        "InstanceID_1",
        "OrgName_1",
        "InstanceID_2",
        "OrgName_2",
        "TrueLabel",
        "PredictedLabel",
        "Probability_Match"
    ]
].copy()

# Use match probability as similarity score for HAC clustering
transformer_hac_score_df = transformer_hac_score_df.rename(
    columns={"Probability_Match": "SimilarityScore"}
)

transformer_hac_score_df.to_csv(
    "bert_scibert_hac_similarity_scores.csv",
    index=False,
    encoding="utf-8-sig"
)